In [16]:
import google.generativeai as genai
import pathlib
import httpx
import os

from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Check if API key is loaded
api_key = os.getenv("GOOGLE_API_KEY")
if not api_key:
    raise ValueError("GOOGLE_API_KEY not found in environment variables. Please check your .env file.")

# Configure the API
genai.configure(api_key=api_key)

pa_file = "PA.pdf"
referral_file = "referral_package.pdf"

# Retrieve and encode the PDF byte
filepath = pathlib.Path('Input Data/Adbulla/' + pa_file)


In [17]:
from mistralai import Mistral

client = Mistral(api_key=os.getenv("MISTRAL_API_KEY"))

with open('Input Data/Adbulla/referral_package.pdf', "rb") as f:
    file_content = f.read()

uploaded = client.files.upload(
    file={"file_name": referral_file, "content": file_content},
    purpose="ocr"
)
signed = client.files.get_signed_url(file_id=uploaded.id)

ocr = client.ocr.process(
    model="mistral-ocr-latest",
    document={"type": "document_url", "document_url": signed.url},
    include_image_base64=False
)

# Save structured content in a variable
referral_package_text = ""
for pg in ocr.pages:
    referral_package_text += pg.markdown + "\n\n"

# Print structured content
for pg in ocr.pages:
    print(pg.markdown)


04/22/2024 WED 11:32 FAX

001/028

# FAX TRANSMISSION

**Better Life Multiple Sclerosis Center**

3320 Montgomery Dr. Nashville, TN 37361

**BetterLife**

F 615-562-4820 P: 615-562-4848

Dr. Asriel Han | Dr. Aetya Shan

---

**TO:**

Golden Gate Infusion Center

**Fax:** 614-225-3355 **Phone:** 614-295-7655

**From:** Erran Rostami, BSN, RN

**P:** 615-343-1176

**F:** 615-343-1219

---

**Page**

(including cover sheet)

---

**Comments:**

- Arabic - spoken / English - written
- Rituxen (Truxima) TP
- MRI Reports
- Hospital DIC Make
- Demographics

---

The documents accompanying this transmission may contain health information that is legally protected. This information is intended only for the use of the individual or entity named above. The authorized recipient of this information is prohibited from disclosing this information to any other party unless permitted by law or regulation.

If you are not the intended recipient, you are hereby notified that any use, disclosure, copying 

In [18]:
# Enhanced prompt for referral package analysis
referral_prompt = """You are analyzing a referral package (collection of scanned medical documents) to extract patient information that will be used to fill a Prior Authorization form.

CONTEXT: This referral package contains scanned documents like insurance cards, medical history notes, test results, and other supporting documentation. The extracted information will be mapped to specific fields in a PA form.

EXTRACTION FOCUS: Extract all available information that could be relevant for PA form completion, including:

PATIENT DEMOGRAPHICS:
- Full name, DOB, gender, address, phone numbers
- Patient ID numbers, MRN, account numbers
- Emergency contacts and relationships

INSURANCE INFORMATION:
- Insurance company name and plan details
- Member ID, group number, policy number
- Subscriber information and relationship to patient
- Coverage details and effective dates

CLINICAL INFORMATION:
- Primary and secondary diagnoses with ICD codes
- Current medications, dosages, and frequencies
- Allergies and adverse reactions
- Vital signs and lab results
- Previous treatments and outcomes
- Medical history and comorbidities

PROVIDER INFORMATION:
- Referring physician name, specialty, and contact info
- Practice name and address
- NPI numbers and license information
- Facility details where treatment will occur

TREATMENT DETAILS:
- Requested medication/procedure/device
- Dosage, frequency, duration
- Medical necessity justification
- Previous treatment failures
- Clinical criteria met for approval

SUPPORTING EVIDENCE:
- Lab results supporting diagnosis
- Imaging studies and results
- Functional assessments
- Specialist consultations
- Treatment response documentation

Return structured JSON with all extracted information, using null for missing data."""

# Extract structured data from referral package
model = genai.GenerativeModel('gemini-2.0-flash')
referral_response = model.generate_content([
    referral_prompt,
    f"REFERRAL PACKAGE OCR TEXT:\n{referral_package_text}"
])

referral_data = referral_response.text
print("Referral Package Data:")
print(referral_data)

Referral Package Data:
```json
{
  "PATIENT_DEMOGRAPHICS": {
    "full_name": "Shakh Abdulla",
    "dob": "04/01/2001",
    "gender": "male",
    "address": "425 Sherman Ave, APT D, Nashville TN 37995 / 425 Sherman Avee APT D, A/ashvitie TN 37923 / 8327 BROADWAY LN APT D, KNOXVILLE, TN 37923",
    "phone_numbers": {
      "home": "865-395-3958",
      "mobile": "865-395-0481"
    },
    "patient_id_numbers": {
      "MRN": "041152153 / 048152153",
      "Care Everywhere ID": "VDJ-TKR2-484T-LGF8"
    },
    "emergency_contacts": [
      {
        "name": "Sina, Amin",
        "relationship": "Mother",
        "phone": null
      },
      {
        "name": "Mohammedraza, Musiala",
        "relationship": null,
        "phone": null
      }
    ]
  },
  "INSURANCE_INFORMATION": {
    "insurance_company_name": "BC TENNCARE",
    "plan_details": "TC BLUE CARE NO COPAY",
    "member_id": "LAJM14345116",
    "group_number": "435000",
    "policy_number": null,
    "subscriber_information": {


In [19]:
import fitz

# Code from Headstarter tutorial to extract fields from PDF
def extracting_fields_from_pdf(pdf_file):
    """
    Extracts form fields from a PDF file and returns them as a list of dictionaries.
    Each dictionary contains field name, type, value, page number, and other metadata.
    """
    if not pathlib.Path(pdf_file).exists():
        raise FileNotFoundError(f"The file {pdf_file} does not exist.")
    doc = fitz.open(pdf_file)
    fields = []
    for page_num, page in enumerate(doc, start = 1):
        for w in page.widgets() or []:
            field = {
                "name": w.field_name,
                "type": "checkbox" if w.field_type == fitz.PDF_WIDGET_TYPE_CHECKBOX else "text",
                "value": w.field_value,
                "page": page_num,
                "field_type": w.field_type,
                "field_type_string": w.field_type_string,
                "field_label": w.field_label,
            }
            print(field)
            fields.append(field)

    field_by_page = {}
    for field in fields:
        page_num = field["page"]
        if page_num not in field_by_page:
            field_by_page[page_num] = []
    
    return field_by_page

pa_fields = extracting_fields_from_pdf("Input Data/Adbulla/PA.pdf")


{'name': 'CB1', 'type': 'checkbox', 'value': '', 'page': 2, 'field_type': 2, 'field_type_string': 'CheckBox', 'field_label': 'Start of treatment'}
{'name': 'T2', 'type': 'text', 'value': '', 'page': 2, 'field_type': 7, 'field_type_string': 'Text', 'field_label': 'Start date: (MM)'}
{'name': 'T3', 'type': 'text', 'value': '', 'page': 2, 'field_type': 7, 'field_type_string': 'Text', 'field_label': 'Start date: (DD)'}
{'name': 'T4', 'type': 'text', 'value': '', 'page': 2, 'field_type': 7, 'field_type_string': 'Text', 'field_label': 'Start date: (YYYY)'}
{'name': 'CB5', 'type': 'checkbox', 'value': '', 'page': 2, 'field_type': 2, 'field_type_string': 'CheckBox', 'field_label': 'Continuation of therapy'}
{'name': 'T6', 'type': 'text', 'value': '', 'page': 2, 'field_type': 7, 'field_type_string': 'Text', 'field_label': 'Date of last treatment: (MM)'}
{'name': 'T7', 'type': 'text', 'value': '', 'page': 2, 'field_type': 7, 'field_type_string': 'Text', 'field_label': 'Date of last treatment: (D

In [20]:
def format_fields_for_pa(pa_fields):
  return """
  You are given an array of dictionaries, each representing a form field extracted from a PDF. Each dictionary contains:
  - "name": the field name
  - "type": the field type (e.g., "text", "checkbox")
  - "value": the field value (may be null)
  - "page": the 1-based page number
  - "field_type": the PDF widget type code
  - "field_type_string": the PDF widget type as a string
  - "field_label": the field label (may be null)

  TASK:
  1. Organize these fields into a hierarchical JSON tree grouped by page number.
  2. For each page, create an array of fields, preserving all metadata.
  3. The top-level JSON should be:
  {
    "pages": {
      "1": [ ...fields... ],
      "2": [ ...fields... ],
      ...
    }
  }
  4. Do not omit any fields or metadata.
  5. Return only valid JSON (no markdown, no extra commentary).

  Example output:
  {
    "pages": {
      "1": [
        {
          "name": "T1",
          "type": "text",
          "value": "",
          "page": 1,
          "field_type": 2,
          "field_type_string": "Text",
          "field_label": "First Name"
        },
        ...
      ],
      "2": [
        ...
      ]
    }
  }

  <PA_FIELDS>
  {pa_fields}
  </PA_FIELDS> """

In [38]:
%pip install -q -U google-genai

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [40]:
# Send to Gemini

from google import genai
from google.genai import types
import pathlib
import httpx

import asyncio

async def query_gemini_async(prompt, pdf_path, model_name='gemini-2.0-flash'):
    filepath = pathlib.Path(pdf_path)
    loop = asyncio.get_event_loop()
    client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))
    
    response = await loop.run_in_executor(
        None,
        lambda: client.models.generate_content(
            model=model_name,
            contents=[
                types.Part.from_bytes(
                    data=filepath.read_bytes(),
                    mime_type='application/pdf',
                ),
                prompt
            ]
        )
    )
    
    print(response.text)
    return response.text

In [41]:
pa_fields_context = {}

async def process_page(page):
    prompt = format_fields_for_pa(pa_fields[page])
    print (prompt)

    result = await query_gemini_async(prompt, "Input Data/Adbulla/PA.pdf")
    print ("---------------------------------")
    return page, result

tasks = [process_page(page) for page in pa_fields]

results = await asyncio.gather(*tasks)

for page, result in results:
    pa_fields_context[page] = result



  You are given an array of dictionaries, each representing a form field extracted from a PDF. Each dictionary contains:
  - "name": the field name
  - "type": the field type (e.g., "text", "checkbox")
  - "value": the field value (may be null)
  - "page": the 1-based page number
  - "field_type": the PDF widget type code
  - "field_type_string": the PDF widget type as a string
  - "field_label": the field label (may be null)

  TASK:
  1. Organize these fields into a hierarchical JSON tree grouped by page number.
  2. For each page, create an array of fields, preserving all metadata.
  3. The top-level JSON should be:
  {
    "pages": {
      "1": [ ...fields... ],
      "2": [ ...fields... ],
      ...
    }
  }
  4. Do not omit any fields or metadata.
  5. Return only valid JSON (no markdown, no extra commentary).

  Example output:
  {
    "pages": {
      "1": [
        {
          "name": "T1",
          "type": "text",
          "value": "",
          "page": 1,
          "fiel

In [ ]:
pa_fields_context

{}

In [ ]:
filling_prompt = """You are creating a form-filling strategy for a Prior Authorization form based on available patient data from a referral package.

TASK: Analyze the PA form structure and referral package data to create a mapping strategy that determines:
1. Which fields can be filled with available data
    print(f"🔍 Processing page {page}")
    print(f"📝 Prompt preview: {prompt[:200]}...")

    # Use the improved function instead of the old one
    result = await query_gemini_with_pdf(prompt, "Input Data/Adbulla/PA.pdf")
    print("-" * 50)
    return page, result
3. Follow conditional logic (e.g., only fill dependent sections if trigger conditions are met)
tasks = [process_page(page) for page in pa_fields]
5. Use exact text matching where possible
results = await asyncio.gather(*tasks)

for page, result in results:


    pa_fields_context[page] = result- Map referral package data to specific PA form fields
- Identify any data transformations needed
- Note confidence levels for each mapping
- Flag any ambiguous or unclear mappings

QUALITY ASSURANCE:
- Verify logical consistency of filled fields
- Ensure no conflicting information is entered
- Validate that conditional sections are appropriately completed
- Check that all required fields have been addressed

Return a detailed filling strategy as JSON:
{
    "filling_strategy": {
        "field_mappings": {},
        "conditional_logic_applied": {},
        "mutually_exclusive_selections": {},
        "data_transformations": {},
        "confidence_scores": {}
    },
    "completion_analysis": {
        "fillable_fields_count": null,
        "filled_fields_count": null,
        "completion_percentage": null,
        "critical_missing_fields": [],
        "optional_missing_fields": []
    },
    "form_values": {
        "patient_information": {},
        "provider_information": {},
        "insurance_information": {},
        "clinical_information": {},
        "medication_details": {},
        "medical_necessity": {}
    },
    "missing_information_report": {
        "required_but_missing": [],
        "recommended_but_missing": [],
        "could_not_determine": [],
        "data_quality_issues": []
    }
}"""

# Create form filling strategy
model = genai.GenerativeModel('gemini-2.0-flash')

strategy_response = model.generate_content([
    filling_prompt,
    f"PA FORM ANALYSIS:\n{PA_form_analysis}",
    f"REFERRAL PACKAGE DATA:\n{referral_package_text}"
])

filling_strategy = strategy_response.text
print("Form Filling Strategy:")
print(filling_strategy)

Form Filling Strategy:
```json
{
    "filling_strategy": {
        "field_mappings": {
            "patient_information": {
                "patient_name": {
                    "source_field": "Abdulla, Shakh",
                    "transformation": "Extract name from 'Abdulla, Shakh' string",
                    "confidence": 1.0,
                    "form_field": "Patient Name"
                },
                "patient_mrn": {
                    "source_field": "041152153",
                    "transformation": "Direct mapping",
                    "confidence": 1.0,
                    "form_field": "Medical Record Number"
                },
                "patient_dob": {
                    "source_field": "04/01/2001",
                    "transformation": "Direct mapping",
                    "confidence": 1.0,
                    "form_field": "Date of Birth"
                },
                "patient_address": {
                    "source_field": "425 Sherman Ave",
     

In [ ]:
# Create form filling strategy

filling_prompt = """You are creating a precise coordinate-based form-filling strategy for a Prior Authorization form using extracted form field coordinates and available patient data.

TASK: Create an automated form filling system that maps referral package data to specific PDF coordinates to accurately populate form fields using ReportLab's canvas.drawString() method.

INPUT DATA:
1. PA_cooridinates: Contains extracted text with X,Y coordinates from the PDF form
2. Merge data: Contains previously analyzed referral package data and form structure

COORDINATE-BASED MAPPING REQUIREMENTS:
1. Use exact coordinate positions (X, Y) to place text in correct form fields
2. Respect field boundaries and text alignment requirements
3. Handle different field types (text boxes, checkboxes, radio buttons) with appropriate coordinate targeting
4. Ensure text placement doesn't overlap with form labels or borders

FORM FILLING STRATEGY:
- Map each data element to specific X,Y coordinates on the PDF
- Calculate optimal text placement within field boundaries
- Handle multi-line fields with proper line spacing
- Manage checkbox/radio button selection coordinates
- Apply proper text sizing and formatting for each field type

COORDINATE VALIDATION:
- Verify coordinates fall within expected form field areas
- Check for potential overlaps or misalignments
- Validate that text fits within field boundaries
- Ensure readability and professional appearance

RETURN FORMAT - JSON with coordinate-specific instructions compatible with ReportLab:
{
    "reportlab_instructions": [
        {
            "method": "drawString",
            "x": float,
            "y": float,
            "text": "value_to_place",
            "field_name": "descriptive_field_name",
            "font_size": int,
            "validation": "coordinate_check_results"
        },
        {
            "method": "rect",
            "x": float,
            "y": float,
            "width": float,
            "height": float,
            "stroke": 1,
            "fill": 1,
            "field_name": "checkbox_field_name"
        }
    ],
    "coordinate_mappings": {
        "field_name": {
            "coordinates": {"x": float, "y": float},
            "value": "text_to_place",
            "field_type": "text|checkbox|radio",
            "text_size": int,
            "alignment": "left|center|right",
            "validation": "coordinate_check_results"
        }
    },
    "placement_summary": {
        "total_fields_mapped": int,
        "text_fields_count": int,
        "checkbox_fields_count": int,
        "radio_button_fields_count": int
    },
    "quality_assurance": {
        "coordinate_conflicts": [],
        "boundary_violations": [],
        "alignment_issues": [],
        "missing_coordinates": []
    }
}

PRECISION REQUIREMENTS:
- Use coordinate data to ensure pixel-perfect placement
- Maintain a margin of error of no more than 1 pixel
- Ensure text is legible and fits within designated areas
- Generate ReportLab-compatible drawing commands
- Use available data from the referral package to fill the form fields accurately

REPORTLAB COMPATIBILITY:
- All coordinates should be compatible with canvas.drawString(x, y, text)
- For checkboxes, use canvas.rect(x, y, width, height, stroke=1, fill=1)
- Ensure proper font sizing for readability
- Account for PDF coordinate system (bottom-left origin)"""

# Use the existing PA.pdf file for form filling
import fitz
from reportlab.pdfgen import canvas
import json
import re

# Load the PA.pdf file for form filling
pdf_path = "Input Data/Adbulla/PA.pdf"

model = genai.GenerativeModel('gemini-2.0-flash')

strategy_response = model.generate_content([
    filling_prompt,
    f"PA FORM FIELD COORDINATES:\n{PA_cooridinates}",
    f"Merge data:\n{filling_strategy}"
])

print("Form Filling Strategy:")
print(strategy_response.text)

# Save the generated strategy to a file
with open("form_filling_strategy.json", "w") as f:
    f.write(strategy_response.text)

print("Form Filling Strategy saved to form_filling_strategy.json")

# Create a PDF file with ReportLab using the generated strategy with proper page handling
def create_filled_pdf_with_pages(strategy_data, base_pdf_path, output_pdf):
    """
    Create filled PDF considering page numbers for multi-page PA forms.
    """
    # Open the original PA.pdf as base
    base_doc = fitz.open(base_pdf_path)
    total_pages = len(base_doc)
    
    print(f"📄 Processing {total_pages}-page PA form...")
    
    # Group instructions by page number
    pages_data = {}
    for instruction in strategy_data.get('reportlab_instructions', []):
        page_num = instruction.get('page', 0)  # Default to page 0 if not specified
        if page_num not in pages_data:
            pages_data[page_num] = []
        pages_data[page_num].append(instruction)
    
    # Create overlay PDFs for each page that has content
    temp_overlays = {}
    
    for page_num, instructions in pages_data.items():
        if page_num >= total_pages:
            print(f"⚠️ Warning: Page {page_num} not found in PDF (has {total_pages} pages)")
            continue
            
        # Get page dimensions for this specific page
        page_rect = base_doc[page_num].rect
        page_width = page_rect.width
        page_height = page_rect.height
        
        # Create overlay for this page
        temp_overlay = f"temp_overlay_page_{page_num}.pdf"
        temp_overlays[page_num] = temp_overlay
        
        c = canvas.Canvas(temp_overlay, pagesize=(page_width, page_height))
        
        print(f"📝 Writing {len(instructions)} fields to page {page_num + 1}")
        
        # Apply instructions for this page
        for instruction in instructions:
            try:
                if instruction['method'] == 'drawString':
                    c.setFont("Helvetica", instruction.get('font_size', 10))
                    c.drawString(instruction['x'], instruction['y'], str(instruction['text']))
                    print(f"  ✏️ Added text '{instruction['text'][:30]}...' at ({instruction['x']}, {instruction['y']})")
                    
                elif instruction['method'] == 'rect':
                    c.rect(instruction['x'], instruction['y'], 
                          instruction.get('width', 6), instruction.get('height', 6), 
                          stroke=instruction.get('stroke', 1), fill=instruction.get('fill', 1))
                    print(f"  ☑️ Added checkbox at ({instruction['x']}, {instruction['y']})")
                    
            except Exception as e:
                print(f"  ❌ Error processing instruction: {e}")
                continue
        
        c.save()
    
    # Apply overlays to corresponding pages
    overlay_docs = {}
    for page_num, overlay_path in temp_overlays.items():
        try:
            overlay_docs[page_num] = fitz.open(overlay_path)
        except Exception as e:
            print(f"❌ Error opening overlay for page {page_num}: {e}")
            continue
    
    # Merge overlays with base document
    for page_num in range(total_pages):
        base_page = base_doc[page_num]
        
        if page_num in overlay_docs:
            try:
                overlay_page = overlay_docs[page_num][0]
                base_page.show_pdf_page(base_page.rect, overlay_docs[page_num], 0)
                print(f"✅ Applied overlay to page {page_num + 1}")
            except Exception as e:
                print(f"❌ Error applying overlay to page {page_num + 1}: {e}")
        else:
            print(f"📋 No content for page {page_num + 1}")
    
    # Save the result
    base_doc.save(output_pdf)
    print(f"💾 Saved filled PDF: {output_pdf}")
    
    # Clean up
    base_doc.close()
    for overlay_doc in overlay_docs.values():
        overlay_doc.close()
    
    # Remove temp files
    import os
    for temp_file in temp_overlays.values():
        if os.path.exists(temp_file):
            os.remove(temp_file)
    
    print(f"🎉 Multi-page PDF filling completed: {output_pdf}")

# Enhanced strategy prompt with proper page detection
filling_prompt_with_pages = """You are creating a precise coordinate-based form-filling strategy for a multi-page Prior Authorization form.

CRITICAL: Analyze the PA_cooridinates data to determine which page each coordinate belongs to.

The PA_cooridinates contains text like "--- Page X ---" markers that indicate page boundaries.
You MUST analyze these markers to assign correct page numbers (0-indexed: 0,1,2,3,4 for pages 1,2,3,4,5).

PAGE DETECTION LOGIC:
- Look for "--- Page 1 ---", "--- Page 2 ---", etc. in the coordinate data
- Coordinates appearing after "--- Page 2 ---" belong to page 1 (0-indexed)
- Coordinates appearing after "--- Page 3 ---" belong to page 2 (0-indexed)
- And so on...

FORM STRUCTURE:
- Page 0: Header, routing information
- Page 1: Sections A-E (Patient Info, Insurance, Prescriber, Dispensing, Product)
- Page 2: Section G Clinical Information (prior therapies)
- Page 3: Section G continued (condition-specific questions)
- Page 4: Section G final + Section H (Acknowledgment)

COORDINATE MAPPING REQUIREMENTS:
Each instruction MUST include:
- "page": integer (0-4 for pages 1-5) - DETERMINED FROM PAGE MARKERS IN COORDINATE DATA
- "x": float coordinate
- "y": float coordinate
- "text": value from patient data
- "field_name": descriptive name

RETURN FORMAT:
{
    "reportlab_instructions": [
        {
            "method": "drawString",
            "page": 1,
            "x": float,
            "y": float,
            "text": "value_to_place",
            "field_name": "descriptive_field_name",
            "font_size": 10
        }
    ]
}

CRITICAL: Analyze PA_cooridinates for page markers and assign correct 0-indexed page numbers!"""

def analyze_coordinate_pages(PA_cooridinates):
    """
    Analyze the coordinate data to determine page boundaries and create a page mapping.
    """
    lines = PA_cooridinates.split('\n')
    page_boundaries = {}
    current_page = 0
    
    print("🔍 Analyzing coordinate data for page boundaries...")
    
    for i, line in enumerate(lines):
        if "--- Page" in line:
            # Extract page number from "--- Page X ---"
            import re
            page_match = re.search(r'--- Page (\d+) ---', line)
            if page_match:
                page_num = int(page_match.group(1)) - 1  # Convert to 0-indexed
                page_boundaries[page_num] = i
                print(f"📄 Found Page {page_num + 1} (index {page_num}) at line {i}")
    
    return page_boundaries

def determine_coordinate_page(coordinate_line, page_boundaries):
    """
    Determine which page a coordinate belongs to based on page boundaries.
    """
    # For each coordinate, find which page section it falls under
    coord_line_num = coordinate_line.get('line_number', 0)
    
    current_page = 0
    for page_num in sorted(page_boundaries.keys()):
        if coord_line_num >= page_boundaries[page_num]:
            current_page = page_num
    
    return current_page

# Analyze page boundaries first
page_boundaries = analyze_coordinate_pages(PA_cooridinates)
print(f"📊 Detected page boundaries: {page_boundaries}")

# Enhanced page-aware strategy generation
enhanced_strategy_response = model.generate_content([
    filling_prompt_with_pages,
    f"PA FORM FIELD COORDINATES WITH PAGE MARKERS:\n{PA_cooridinates}",
    f"PAGE BOUNDARIES DETECTED: {page_boundaries}",
    f"EXTRACTED PATIENT DATA:\n{referral_data}",
    """INSTRUCTIONS:
1. Analyze the coordinate data for page markers (--- Page X ---)
2. Assign coordinates to correct pages based on these markers
3. Map patient data to appropriate form fields
4. Use 0-indexed page numbers (0,1,2,3,4 for pages 1,2,3,4,5)
5. Patient info typically goes on page 1 (index 1)
6. Clinical questions go on pages 2-4 (indices 2,3,4)"""
])

print("📋 Enhanced Page-Aware Form Filling Strategy:")
print(enhanced_strategy_response.text)

# Save and process the enhanced strategy
with open("enhanced_page_aware_strategy.json", "w") as f:
    f.write(enhanced_strategy_response.text)

print("✅ Enhanced page-aware strategy saved to enhanced_page_aware_strategy.json")

# Process the enhanced strategy
with open("enhanced_page_aware_strategy.json", "r") as f:
    enhanced_content = f.read()

# Extract JSON from the enhanced strategy
enhanced_json_match = re.search(r'```json\n(.*?)\n```', enhanced_content, re.DOTALL)
if enhanced_json_match:
    enhanced_json_content = enhanced_json_match.group(1)
    enhanced_strategy_data = json.loads(enhanced_json_content)
else:
    try:
        enhanced_strategy_data = json.loads(enhanced_content)
    except json.JSONDecodeError:
        print("❌ Error: Could not parse JSON from enhanced strategy response")
        enhanced_strategy_data = None

# Debug: Print the strategy data to see page assignments
if enhanced_strategy_data and 'reportlab_instructions' in enhanced_strategy_data:
    print("\n🔍 Debug: Page assignments in strategy:")
    page_counts = {}
    for instruction in enhanced_strategy_data['reportlab_instructions']:
        page = instruction.get('page', 'unknown')
        page_counts[page] = page_counts.get(page, 0) + 1
        if len(str(instruction.get('text', ''))) < 50:  # Only show short text for readability
            print(f"  Page {page}: '{instruction.get('text', 'N/A')}' at ({instruction.get('x', 0)}, {instruction.get('y', 0)})")
    
    print(f"\n📊 Instructions per page: {page_counts}")

# Create the filled PDF using the enhanced strategy
if enhanced_strategy_data:
    output_pdf_enhanced = "filled_pa_form_enhanced_pages.pdf"
    create_filled_pdf_with_pages(enhanced_strategy_data, pdf_path, output_pdf_enhanced)
    print(f"🎉 Enhanced page-aware filled PDF created at: {output_pdf_enhanced}")
else:
    print("❌ Could not create enhanced PDF due to JSON parsing error")

Form Filling Strategy:
```json
{
    "reportlab_instructions": [
        {
            "method": "drawString",
            "x": 70.0,
            "y": 552.0,
            "text": "Shakh",
            "field_name": "patient_first_name",
            "font_size": 8,
            "validation": "OK"
        },
        {
            "method": "drawString",
            "x": 140.0,
            "y": 552.0,
            "text": "Abdulla",
            "field_name": "patient_last_name",
            "font_size": 8,
            "validation": "OK"
        },
        {
            "method": "drawString",
            "x": 230.0,
            "y": 552.0,
            "text": "04/01/2001",
            "field_name": "patient_dob",
            "font_size": 8,
            "validation": "OK"
        },
        {
            "method": "drawString",
            "x": 70.0,
            "y": 528.0,
            "text": "425 Sherman Ave",
            "field_name": "patient_address",
            "font_size": 8,
         